In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import re
import pandas as pd
import numpy as np
import spacy
import sys
import subprocess
import nltk
from nltk.corpus import stopwords

# Download required data
nltk.download('stopwords', quiet=True)


# Robust spaCy Model Installation 
MODEL_NAME = "en_core_web_sm"

def ensure_spacy_model():
    try:
        nlp = spacy.load(MODEL_NAME)
        print(f"✓ {MODEL_NAME} already installed and loaded.")
        return nlp
    except OSError:
        print(f"Model '{MODEL_NAME}' not found. Installing now...")
        try:
            subprocess.check_call([sys.executable, "-m", "spacy", "download", MODEL_NAME])
            nlp = spacy.load(MODEL_NAME)
            print(f"✓ Successfully installed and loaded {MODEL_NAME}.")
            return nlp
        except Exception as e:
            print(f"Auto-install failed: {e}")
            print("Please run this command manually in your terminal:")
            print(f"    python -m spacy download {MODEL_NAME}")
            print("Then restart the notebook kernel.")
            raise

# Execute once
nlp = ensure_spacy_model()

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Stopwords setup
DOMAIN_STOPWORDS = {'film', 'movie', 'story', 'character', 'director', 'actor', 'watch'}
NLTK_STOPWORDS = set(stopwords.words('english'))
ALL_STOPWORDS = NLTK_STOPWORDS.union(DOMAIN_STOPWORDS)

def protect_compound_names(text: str) -> str:
    """
    Replace spaces in common compound names with underscores.
    This prevents lemmatization from breaking proper names.
    """
    # Common director patterns (expand based on your data)
    name_patterns = [
        (r'christopher nolan', 'christopher_nolan'),
        (r'quentin tarantino', 'quentin_tarantino'),
        (r'martin scorsese', 'martin_scorsese'),
        (r'steven spielberg', 'steven_spielberg'),
        (r'david fincher', 'david_fincher'),
        (r'james cameron', 'james_cameron'),
        (r'ridley scott', 'ridley_scott'),
        (r'peter jackson', 'peter_jackson'),
        (r'coen brothers', 'coen_brothers'),
        (r'wes anderson', 'wes_anderson'),
    ]
    
    # Apply all patterns
    for pattern, replacement in name_patterns:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    
    return text

def clean_text(text: str) -> str:
    """
    Complete text cleaning pipeline as specified in Section 4.1.
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Step 3: Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Step 4: PROTECT COMPOUND NAMES (CRITICAL STEP)
    text = protect_compound_names(text)
    
    # Step 5: Remove special characters (keep letters, numbers, spaces, underscores)
    text = re.sub(r'[^a-z0-9\s_]', '', text)
    
    # Step 6: Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 7: Tokenize with spaCy for lemmatization
    doc = nlp(text)
    
    # Step 8: Remove stopwords and lemmatize
    tokens = []
    for token in doc:
        if token.text not in ALL_STOPWORDS and not token.is_punct:
            # If token contains underscore, it's a protected name - keep as-is
            if '_' in token.text:
                tokens.append(token.text)
            else:
                tokens.append(token.lemma_)
    
    # Step 9: Rejoin
    return ' '.join(tokens)

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

#### Wikipedia Plot Truncation (200 WORDS, not characters)


In [ ]:
def preprocess_for_cbf(row) -> str:
    """
    Build combined_content exactly as specified in Section 4.2.
    """
    components = []
    
    # 1. Directors (4×)
    directors = row.get('directors', [])
    if isinstance(directors, list) and directors:
        components.extend([str(d).lower() for d in directors] * 4)
    
    # 2. Lead actor (3×)
    cast = row.get('cast_top5', [])
    if isinstance(cast, list) and cast:
        components.extend([str(cast[0]).lower()] * 3)
        
        # 3. 2nd actor (2×)
        if len(cast) > 1:
            components.extend([str(cast[1]).lower()] * 2)
        
        # 4. 3rd, 4th, 5th actors (1× each)
        for i in range(2, min(5, len(cast))):
            components.append(str(cast[i]).lower())
    
    # 5. Writers (2×)
    writers = row.get('writers', [])
    if isinstance(writers, list) and writers:
        components.extend([str(w).lower() for w in writers] * 2)
    
    # 6. Plot keywords (3×)
    keywords = row.get('plot_keywords', '')
    if isinstance(keywords, str) and keywords:
        keyword_list = [k.strip().lower() for k in keywords.split('|') if k.strip()]
        components.extend(keyword_list * 3)
    
    # 7. Genres (3×)
    genres = row.get('genres_list', [])
    if isinstance(genres, list) and genres:
        components.extend([str(g).lower() for g in genres] * 3)
    
    # 8. Overview (1×)
    overview = row.get('overview', '')
    if isinstance(overview, str) and overview:
        components.append(overview.lower())
    
    # 9. Wikipedia plot - FIRST 200 WORDS (1×) - CRITICAL FIX
    wiki = row.get('wiki_plot', '')
    if isinstance(wiki, str) and wiki:
        words = wiki.split()[:200]  # ← 200 WORDS, not characters
        components.append(' '.join(words).lower())
    
    # 10. Production company (1×)
    prod_companies = row.get('production_companies_list', [])
    if isinstance(prod_companies, list) and prod_companies:
        components.append(str(prod_companies[0]).lower())
    
    # 11. Original language (2×)
    lang = row.get('original_language', '')
    if isinstance(lang, str) and lang:
        components.extend([lang.lower()] * 2)
    
    # 12. Content rating (1×)
    rating = row.get('content_rating', '')
    if isinstance(rating, str) and rating:
        components.append(rating.lower())
    
    # Join all components
    raw_text = ' '.join(components)
    
    # Apply cleaning pipeline
    return clean_text(raw_text)

#### Feature Engineering

In [ ]:
import ast

print("=" * 40)
print("BUILDING COMBINED CONTENT FOR CBF")
PRINT("=" * 40)


# movies = pd.read_csv("../data/processed/movies_merged.csv")
movies = pd.read_csv('/content/drive/MyDrive/datasets/movies_merged.csv')

# Convert string representations of lists back to actual lists
for col in ['genres_list', 'cast_top5', 'directors', 'writers', 'production_companies_list']:
    if col in movies.columns and movies[col].dtype == 'object':
        movies[col] = movies[col].apply(lambda x: ast.literal_eval(x) if pd.notna(x) and x.startswitch('[') else [])

print(f"\nLoaded {len(movies)} movies")
print(f"Sample movies: {movies['title'].iloc[0]}")

# Apply preprocessing
print("\nBuilding combined_content (this may take 5-10 minutes)...")
movies = ['combined_content'] = movies.apply(preprocess_for_cbf, axis=1)



print("=" * 40)
print("COMBINED CONTENT STATISTICS")
print("=" * 60)

content_stats = {
    'Movies with content': (movies['combined_content'].str.len() > 0).sum(),
    'Movies without content': (movies['combined_content'].str.len() == 0).sum(),
    'Average length (chars)': movies['combined_content'].str.len().mean(),
    'Median length (chars)': movies['combined_content'].str.len().median(),
    'Min length': movies['combined_content'].str.len().min(),
    'Max length': movies['combined_content'].str.len().max()
}

for key, value in content_stats.items():
    print(f"{key:25}: {value:,.0f}")

print("\n" + "=" * 40)
print("SAMPLE COMBINED CONTENT")
print("=" * 60)

# Show first 3 movies
for idx in range(min(3, len(movies))):
    title = movies.iloc[idx]['title']
    content = movies.iloc[idx]['combined_content']
    print(f"\n--- {title} ---")
    print(f"Length: {len(content)} chars")
    print(f"Preview: {content[:500]}...")
    print("-" * 40)

# Save for CBF training
print("\n" + "=" * 60)
print("SAVING OUTPUTS")
print("=" * 60)

# Save as CSV (for readability)
movies.to_csv('../data/processed/movie_content.csv', index=False)
# movies.to_csv('/content/drive/MyDrive/movie_content.csv', index=False)
print("✓ Saved: ../data/processed/movie_content.csv")


# Save as pickle (for fast loading with preserved types)
movies.to_pickle('../data/processed/movies_df.pkl')
# movies.to_pickle('/content/drive/MyDrive/movies_df.pkl')
print("✓ Saved: ../data/processed/movies_df.pkl")

print(f"\nFinal shape: {movies.shape}")
print(f"Columns: {movies.columns.tolist()}")


In [ ]:
# Check if compound names are being protected
toy_story = movies[movies['id'] == 862].iloc[0]
print("Original director:", toy_story['directors'])
print("In combined_content:", ' '.join([w for w in toy_story['combined_content'].split() if 'lasseter' in w][:4]))

In [ ]:
# Check Christopher Nolan in Inception
inception = movies[movies['id'] == 27205].iloc[0] if 27205 in movies['id'].values else None
if inception is not None:
    print(f"Inception director: {inception['directors']}")
    nolan_tokens = [w for w in inception['combined_content'].split() if 'nolan' in w][:4]
    print(f"In combined_content: {nolan_tokens}")